# Getting Started with NEUIToolkit

This notebook demonstrates basic usage of NEUIToolkit for knowledge extraction from documents.

## Prerequisites

1. Install dependencies: `pip install -r requirements.txt`
2. Configure LLM provider in `.env` file
3. (Optional) Set up Neo4j database

## What You'll Learn

- How to extract knowledge from a single document
- How to use the quality assurance layer
- How to visualize extraction results
- How to export to Neo4j

## Setup

First, let's import the necessary modules and configure our environment.

In [ ]:
import sys
import os
from pathlib import Path
import json

# Add parent directory to path
sys.path.append(str(Path.cwd().parent))

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Import NEUIToolkit modules
from backend.orchestrator import (
    read_corpus,
    run_entity_pass,
    run_relationship_pass,
    run_rule_pass
)
from backend.quality_assurance import QualityAssurance
from llm.llm_utils import get_provider_stats

print("✅ NEUIToolkit modules loaded successfully!")

## Step 1: Load a Sample Document

Let's create a sample text document to extract knowledge from.

In [ ]:
# Sample educational text about biology
sample_text = """
Cells are the basic structural and functional units of all living organisms. 
The cell membrane surrounds the cell and controls the movement of substances in and out.

Mitochondria are known as the powerhouse of the cell because they produce ATP through 
cellular respiration. The nucleus contains the cell's genetic material (DNA) and controls 
cell activities.

Plant cells have a cell wall that provides structural support. Chloroplasts in plant cells 
perform photosynthesis to convert light energy into chemical energy.

If a cell lacks sufficient mitochondria, then it will have reduced energy production.
If photosynthesis occurs, then glucose is produced.
"""

print("Sample document:")
print("=" * 80)
print(sample_text)
print("=" * 80)
print(f"\nDocument length: {len(sample_text)} characters")

## Step 2: Extract Entities

Extract key concepts and entities from the text.

In [ ]:
print("Extracting entities...\n")

entities = run_entity_pass(sample_text)

print(f"✅ Extracted {len(entities)} entities:\n")
for i, entity in enumerate(entities, 1):
    print(f"{i}. {entity['name']}")
    print(f"   Category: {entity.get('category', 'N/A')}")
    print(f"   Aliases: {', '.join(entity.get('aliases', []))}")
    print()

## Step 3: Extract Relationships

Extract semantic relationships between entities.

In [ ]:
print("Extracting relationships...\n")

relationships = run_relationship_pass(sample_text)

print(f"✅ Extracted {len(relationships)} relationships:\n")
for i, rel in enumerate(relationships, 1):
    print(f"{i}. {rel['subject']} → {rel['predicate']} → {rel['object']}")
    if 'justification' in rel:
        print(f"   Justification: {rel['justification']}")
    print()

## Step 4: Extract Rules

Extract logical if-then rules from the text.

In [ ]:
print("Extracting rules...\n")

rules = run_rule_pass(sample_text)

print(f"✅ Extracted {len(rules)} rules:\n")
for i, rule in enumerate(rules, 1):
    print(f"{i}. IF: {rule.get('if', 'N/A')}")
    print(f"   THEN: {rule.get('then', 'N/A')}")
    print(f"   Confidence: {rule.get('confidence', 'N/A')}")
    print()

## Step 5: Quality Assessment

Use the quality assurance layer to assess extraction quality.

In [ ]:
# Initialize QA system
qa = QualityAssurance(
    min_confidence=0.5,
    enable_strict_mode=False
)

# Assess entities
filtered_entities, entity_metrics = qa.filter_low_quality(entities, 'entity')

print("Entity Quality Assessment:")
print("=" * 80)
print(f"Total entities: {entity_metrics['total_count']}")
print(f"Passed QA: {entity_metrics['passed_count']}")
print(f"Failed QA: {entity_metrics['failed_count']}")
print(f"Quality Score: {entity_metrics['quality_score']:.3f}")
print()

# Assess relationships
filtered_relationships, rel_metrics = qa.filter_low_quality(relationships, 'relationship')

print("Relationship Quality Assessment:")
print("=" * 80)
print(f"Total relationships: {rel_metrics['total_count']}")
print(f"Passed QA: {rel_metrics['passed_count']}")
print(f"Failed QA: {rel_metrics['failed_count']}")
print(f"Quality Score: {rel_metrics['quality_score']:.3f}")
print()

# Assess rules
filtered_rules, rule_metrics = qa.filter_low_quality(rules, 'rule')

print("Rule Quality Assessment:")
print("=" * 80)
print(f"Total rules: {rule_metrics['total_count']}")
print(f"Passed QA: {rule_metrics['passed_count']}")
print(f"Failed QA: {rule_metrics['failed_count']}")
print(f"Quality Score: {rule_metrics['quality_score']:.3f}")

## Step 6: Generate Overall Quality Report

In [ ]:
# Generate comprehensive quality report
report = qa.generate_quality_report(
    entity_metrics,
    rel_metrics,
    rule_metrics
)

print("Overall Quality Report:")
print("=" * 80)
print(f"Overall Quality Score: {report['overall_quality_score']:.3f}")
print(f"\nRecommendations:")
for rec in report['recommendations']:
    print(f"  - {rec}")
print(f"\nSummary: {report['summary']}")

## Step 7: Visualize Knowledge Graph

Create a simple visualization of the extracted knowledge.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

# Create graph
G = nx.DiGraph()

# Add entities as nodes
for entity in filtered_entities:
    G.add_node(entity['name'], category=entity.get('category', 'Unknown'))

# Add relationships as edges
for rel in filtered_relationships:
    G.add_edge(
        rel['subject'],
        rel['object'],
        predicate=rel['predicate']
    )

# Visualize
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=2, iterations=50)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=2000, alpha=0.9)

# Draw edges
nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20, width=2)

# Draw labels
nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold')

# Draw edge labels (predicates)
edge_labels = nx.get_edge_attributes(G, 'predicate')
nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=8)

plt.title("Extracted Knowledge Graph", fontsize=16, fontweight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

print(f"\nGraph Statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Density: {nx.density(G):.3f}")

## Step 8: Check LLM Provider Statistics

In [ ]:
# Get provider statistics
stats = get_provider_stats()

print("LLM Provider Statistics:")
print("=" * 80)
print(f"Primary Provider: {stats['primary_provider']}")
print(f"Total Calls: {stats['total_calls']}")
print(f"Total Cost: ${stats['total_cost']:.4f}")
print(f"Successful Calls: {stats['successful_calls']}")
print(f"Failed Calls: {stats['failures']}")

## Step 9: Export Results (Optional)

Save the extracted knowledge to JSON files.

In [ ]:
# Create output directory
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

# Save entities
with open(output_dir / "entities.json", "w") as f:
    json.dump(filtered_entities, f, indent=2)

# Save relationships
with open(output_dir / "relationships.json", "w") as f:
    json.dump(filtered_relationships, f, indent=2)

# Save rules
with open(output_dir / "rules.json", "w") as f:
    json.dump(filtered_rules, f, indent=2)

# Save quality report
with open(output_dir / "quality_report.json", "w") as f:
    json.dump(report, f, indent=2)

print(f"✅ Results saved to {output_dir}/")
print(f"   - entities.json ({len(filtered_entities)} entities)")
print(f"   - relationships.json ({len(filtered_relationships)} relationships)")
print(f"   - rules.json ({len(filtered_rules)} rules)")
print(f"   - quality_report.json")

## Next Steps

Now that you've seen the basics, try:

1. **Process your own documents**: Use `read_corpus()` to load PDF/DOCX files
2. **Adjust QA thresholds**: Experiment with different confidence levels
3. **Export to Neo4j**: See the advanced notebook for Neo4j integration
4. **Batch processing**: Process multiple documents in parallel
5. **Custom prompts**: Modify prompt templates for domain-specific extraction

## Additional Resources

- [README.md](../README.md) - Full documentation
- [ARCHITECTURE.md](../ARCHITECTURE.md) - System architecture
- [CONTRIBUTING.md](../CONTRIBUTING.md) - Development guide
- Example 02: Advanced workflows with Neo4j
- Example 03: Batch processing and optimization